### Cálculo do índice de reclamações por entidade

É realizada a junção das bases de **Reclamações e Clientes**, permitindo analisar a quantidade de reclamações em relação ao número de clientes de cada entidade.

Esse indicador é calculado dividindo o total de reclamações procedentes pelo total de clientes.

**Importante**: o indicador é cálculado sempre um mês atrás, ou seja, M-1. Caso o mês atual for abril, o mês calculado será março.

**Fórmula utilizada:**

`Índice de reclamações = Total de reclamações procedentes / Total de clientes`

O objetivo é acompanhar a proporção de reclamações em relação à quantidade de clientes de cada entidade, permitindo identificar quais apresentam maiores índices e que, portanto, podem necessitar de maior atenção.

In [0]:
from pyspark.sql.functions import col, to_date, add_months, current_date, trunc

In [0]:
entidades = spark.table(
    "workspace.default.clientes_empresa"
).select("entidade").distinct()
display(entidades)

In [0]:
%sql
SELECT * FROM workspace.default.reclamacoes_empresa

In [0]:
df = spark.table(
    "workspace.default.reclamacoes_empresa"
)

reclamacoes = df.withColumn(
    "data_fechamento",
    to_date(col("data_fechamento"), "yyyyMM")
).filter(
    df["tipo_reclamacao"] == 'PROCEDENTES'
).groupBy(
    "entidade", 
    "data_fechamento"
).count().orderBy(
    "entidade",
    "data_fechamento"
).withColumnRenamed("count", "total_reclamacoes").filter(
    col("data_fechamento") == add_months(trunc(current_date(), "MM"), -1)
)

display(reclamacoes)

In [0]:
%sql 
SELECT * FROM workspace.default.clientes_empresa

In [0]:
dg = spark.table(
    "workspace.default.clientes_empresa"
)

clientes = dg.dropDuplicates(["cpf_cliente"]).withColumn(
    "data_cadastro",
    to_date(col("data_cadastro"), "yyyyMM")
).withColumn(
    "fechamento_cadastro",
    to_date(col("fechamento_cadastro"), "yyyyMM")
).filter(
    dg["fechamento_cadastro"].isNull()
).filter(
    col("data_cadastro") <= add_months(trunc(current_date(), "MM"), -1)
).groupBy(
    "entidade"
).count().orderBy(
    "entidade"
).withColumnRenamed("count", "total_clientes")

display(clientes)

In [0]:
satisfacao_empresa = reclamacoes.join(
    clientes,
    on="entidade",
    how="left"
).withColumn(
    "vlr_porc_reclam", 
    (col("total_reclamacoes") / col("total_clientes"))
)

display(satisfacao_empresa)